# Full-Shot 'Transport' Prediction 🔮

>This is one of the more complicated modules presently implemented in POPSIM. It is recommended to read the following notebooks before proceeding:
>- `intro_to_modules.ipynb`
>- `training_td_modules.ipynb`
>- `profile_predictor.ipynb`

## Introduction

1D profiles of temperature and density can be used to determine many important aspects of plasma behavior, such as MHD stability and expected fusion power output. Therefore, being able to accurately predict these profiles before running a shot will be beneficial for pulse planning and operation. The goal of the `TransportPredictor` module is to predict the time evolution of these profiles given only the initial conditions and a planned trajectory from actuators that will be available in a future tokamak power plant such as plasma current, shaping, input power, and fueling.

This notebook will walk through the structure of the `TransportPredictor` module, show some intermediate results, and outline the training procedure.

The fueling treatment is a bit hand-wavy at the moment. A brief overview of the present implementation of the module is given in section VI of [A.M. Wang et al., 2025](https://arxiv.org/abs/2509.10244), and a more comprehensive analysis is in preparation.

## Module structure

The `TransportPredictor` module is composed of four submodules:
1. `ProfilePredictor`: Outputs profiles of $T_e$ and $n_e$. Same as described in profile_predictor.ipynb
2. `PowerBalance`: Predicts evolution of $W_{tot}$. A modified version of the example in training_td_modules.ipynb
3. `OhmicPower`: Predicts $P_{oh}$
4. `RadiatedPower`: Predicts $P_{rad}$

Note that the `PowerBalance` module is the only component that is time-dependent. The remaining modules use the output $W_{tot}$ to make their predictions at a given timestep.

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.display import Markdown, display
from popsim.notebook_utils import print_class_without_methods
from popsim.modules.transport_predictor.module import TransportPredictor

display(Markdown(f"```python\n{print_class_without_methods(TransportPredictor, ignored_methods={'__init__', 'init', '__call__'})}\n```"))

# 1. Running the TransportPredictor

The following demonstrations use pre-trained module checkpoints because the training procedure for this module is a bit involved, takes a long time, and is not well-suited for notebooks. An identical copy of the `tcv_11` checkpoint can be obtained by following the training procedure as outlined in section 2, or by running the script at `modules/transport_predictor/transport_sample_checkpoint.py`. Data sharing agreements prevent us from publishing a more comprehensive TCV dataset for now, but we have included additional model checkpoints for comparison.

>We have not yet gotten permission to upload a real sample of TCV data. We aim to do so in the near-ish future.
>The scrambled dataset is included to demonstrate the mechanics of the dataloading pipeline.


Since the `TransportPredictor` is composed of multiple submodules, its config contains the configs for each submodule.
To help organize the configs and ensure there aren't accidental side-effects from modifying dictionaries, the main module's config is implemented using `TrainConfig`. This is a thin wrapper around pydantic's `BaseModel` which only allows deep copies.

## 1.1 Restore module and run evals

In [ ]:
import os
import shutil
import xarray as xr
from popsim import DATA_DIR, PACKAGE_ROOT
from popsim.ml.trainer import Trainer
from popsim.modules.transport_predictor.train_configs import BASE_CONFIG
from popsim.modules.transport_predictor.training_run_builder import TransportPredictorTrainRunBuilder

# Load the base config and point it to the checkpoint directory and dataset
dataset = "tcv_11"  # Change this to the checkpoint you want to load
checkpoint_zip = os.path.join(PACKAGE_ROOT, "checkpoints", "transport_predictor_demo", f"sample_checkpoint_{dataset}.zip")
checkpoint_dir = os.path.join(PACKAGE_ROOT, "checkpoints", "transport_predictor_demo", f"sample_checkpoint_{dataset}")

ds_path = os.path.join(DATA_DIR, "tcv", "scrambled_transport_sample.nc")
test_shots = xr.open_dataset(ds_path).shot.values.tolist()
cheat_training = True if dataset == "tcv_11" else False  # dataset too small to do a proper train/test split, training and testing on same data (NOT ALLOWED, THIS IS FOR DEMO PURPOSES ONLY)

shutil.unpack_archive(checkpoint_zip, checkpoint_dir)
transport_predictor_config = BASE_CONFIG.model_copy(
    update={
        "checkpoint_dir": checkpoint_dir,
        "dataloader_config": {
            **BASE_CONFIG.dataloader_config,
            "ds_path": ds_path,
            "test_shots": test_shots,
            "cheat_training": cheat_training,
        },
        "model_init_config": {
            **BASE_CONFIG.model_init_config,
            "restore_submodules": False,  # Don't need to restore submodules because we're restoring the whole model
        }
    }
)

# Build the trainer from the config and restore the best checkpoint
_, train_dl, _, test_dl = TransportPredictorTrainRunBuilder.get_dataloaders(transport_predictor_config.dataloader_config)
trainer = Trainer(
    model=TransportPredictorTrainRunBuilder.model_init(train_dl, transport_predictor_config.model_init_config),
    loss_fn=TransportPredictorTrainRunBuilder.get_loss_fn(transport_predictor_config.loss_config),
    optimizer=TransportPredictorTrainRunBuilder.get_optimizer(transport_predictor_config.optimizer_config),
    checkpoint_dir=transport_predictor_config.checkpoint_dir,
)
trainer.restore_best_checkpoint()

# Run evaluation on the test set and extract the input and output datasets
eval_data = trainer.run_evals(test_dl)
input_ds = eval_data.input_ds.reset_index("sample")
output_ds = eval_data.output_ds.reset_index("sample")

## 1.2 Measured vs Predicted Profiles

Scrollable plot of predicted vs measured profiles across different shots.
Dark-dashed colors represent measured values, bright-solid colors represent predictions.

In [ ]:
import xarray as xr
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import IntSlider, VBox
import numpy as np

# Rename dimensions for consistency
ds_profile = xr.merge([input_ds, output_ds], compat="no_conflicts", join="exact")

# Define colors for different rho values
rho_colors = {
    0: ("firebrick", "red"),
    0.4: ("darkorange", "orange"),
    0.6: ("olive", "yellow"),
    0.8: ("green", "lime"),
    0.9: ("darkturquoise", "cyan"),
    1.0: ("blueviolet", "violet"),
}

shot_list = sorted([int(s) for s in ds_profile.shot.values])[2:]

# Create a FigureWidget that can be updated in-place
fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Electron Density', 'Electron Temperature'
), vertical_spacing=0.12)

# Convert to FigureWidget for dynamic updates
fig_widget = go.FigureWidget(fig)
fig_widget.update_layout(height=700, width=1000, legend=dict(x=1.02, y=1))
fig_widget.update_xaxes(title_text='Time [s]', range=[0, 1.5])
fig_widget.update_yaxes(title_text='Electron Density [10^20 m^-3]', row=1, col=1)
fig_widget.update_yaxes(title_text='Electron Temperature [keV]', row=2, col=1)

def update_profile_plot(change):
    shot_idx = shot_slider.value
    shot = shot_list[shot_idx]
    shot_data = ds_profile.where(ds_profile.shot == shot, drop=True).squeeze()
    time_vals = shot_data["time"].values
    
    # Clear existing traces
    fig_widget.data = []
    
    # Update subplot titles
    fig_widget.layout.annotations[0].text = f'Shot {shot}: Electron Density'
    fig_widget.layout.annotations[1].text = f'Shot {shot}: Electron Temperature'
    
    for rho_val, (target_color, pred_color) in rho_colors.items():

        
        # Electron density
        ne_target = shot_data["ne20_rho"].sel(rho=rho_val, method="nearest")
        ne_predicted = shot_data["output.profile_predictor_output.ne"].sel(rho=rho_val, method="nearest")
        fig_widget.add_trace(go.Scatter(x=time_vals, y=ne_target, mode='lines', name=f'ρ={rho_val} Measured',
                                  line=dict(color=target_color, dash='dash', width=2)), row=1, col=1)
        fig_widget.add_trace(go.Scatter(x=time_vals, y=ne_predicted, mode='lines', name=f'ρ={rho_val} Predicted',
                                  line=dict(color=pred_color, width=2)), row=1, col=1)
        
        # Electron temperature
        te_target = shot_data["Te_keV_rho"].sel(rho=rho_val, method="nearest")
        te_predicted = shot_data["output.profile_predictor_output.te"].sel(rho=rho_val, method="nearest")
        fig_widget.add_trace(go.Scatter(x=time_vals, y=te_target, mode='lines', name=f'ρ={rho_val} Measured',
                                  line=dict(color=target_color, dash='dash', width=2), showlegend=False), row=2, col=1)
        fig_widget.add_trace(go.Scatter(x=time_vals, y=te_predicted, mode='lines', name=f'ρ={rho_val} Predicted',
                                  line=dict(color=pred_color, width=2), showlegend=False), row=2, col=1)

shot_slider = IntSlider(min=0, max=len(shot_list)-1, step=1, value=0, description='Shot Index')
shot_slider.observe(update_profile_plot, names='value')

# Initialize with first shot
update_profile_plot(None)

# Display slider and figure
VBox([shot_slider, fig_widget])

## 1.3 Power Balance Module

We can also take a look at what the other submodules are doing.

While the outputs of submodules are included in the cost function of the `TransportPredictor` to ensure they are somewhat realistic, they are weighted significantly less than the profiles so there are potentially large disagreements.

In particular, the measured $P_{oh}$ and $P_{rad}$ are subject to large uncertainties.

In [ ]:
import xarray as xr
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import IntSlider, VBox
import numpy as np

ds_power = xr.merge([input_ds, output_ds], compat="no_conflicts", join="exact")

shot_list_power = sorted([int(s) for s in ds_power.shot.values])[2:]

# Create a FigureWidget for dynamic updates
fig_power = make_subplots(rows=2, cols=1, subplot_titles=(
    'Stored Energy', 'Power Sources'
), vertical_spacing=0.12)

fig_power_widget = go.FigureWidget(fig_power)
fig_power_widget.update_layout(height=700, width=1000)
fig_power_widget.update_xaxes(title_text='Time [s]')
fig_power_widget.update_yaxes(title_text='Stored Energy [MJ]', row=1, col=1)
fig_power_widget.update_yaxes(title_text='Power [MW]', row=2, col=1)

def update_power_plot(change):
    shot_idx = power_slider.value
    shot = shot_list_power[shot_idx]
    shot_data = ds_power.where(ds_power.shot == shot, drop=True).squeeze()
    time_vals = shot_data["time"].values
    
    # Clear existing traces
    fig_power_widget.data = []
    
    # Update subplot titles
    fig_power_widget.layout.annotations[0].text = f'Shot {shot}: Stored Energy'
    fig_power_widget.layout.annotations[1].text = f'Shot {shot}: Power Sources'
    
    # Stored Energy plot
    stored_energy_target = shot_data["Wtot_MJ"].values
    stored_energy_predicted = shot_data["state.power_balance_state.Wtot_MJ"].values
    
    fig_power_widget.add_trace(go.Scatter(x=time_vals, y=stored_energy_target, mode='lines', name='Target',
                          line=dict(color='purple', dash='dash', width=2)), row=1, col=1)
    fig_power_widget.add_trace(go.Scatter(x=time_vals, y=stored_energy_predicted, mode='lines', name='Predicted',
                          line=dict(color='pink', width=2)), row=1, col=1)
    
    # Power Sources plot
    p_oh_target = shot_data["P_oh_MW"].values
    p_oh_predicted = shot_data["output.p_oh_output.P_oh_MW_pred"].values
    p_rad_target = shot_data["P_rad_MW"].values
    p_rad_predicted = shot_data["output.p_rad_output.P_rad_MW_pred"].values
    
    fig_power_widget.add_trace(go.Scatter(x=time_vals, y=p_oh_target, mode='lines', name='P_oh Target',
                          line=dict(color='blue', dash='dash', width=2)), row=2, col=1)
    fig_power_widget.add_trace(go.Scatter(x=time_vals, y=p_oh_predicted, mode='lines', name='P_oh Predicted',
                          line=dict(color='cyan', width=2)), row=2, col=1)
    fig_power_widget.add_trace(go.Scatter(x=time_vals, y=p_rad_target, mode='lines', name='P_rad Target',
                          line=dict(color='red', dash='dash', width=2)), row=2, col=1)
    fig_power_widget.add_trace(go.Scatter(x=time_vals, y=p_rad_predicted, mode='lines', name='P_rad Predicted',
                          line=dict(color='orange', width=2)), row=2, col=1)

power_slider = IntSlider(min=0, max=len(shot_list_power)-1, step=1, value=0, description='Shot Index')
power_slider.observe(update_power_plot, names='value')

# Initialize with first shot
update_power_plot(None)

# Display slider and figure
VBox([power_slider, fig_power_widget])

# 2. Training the TransportPredictor

> The following cells are for illustration purposes only. I do **NOT** recommend attempting to run the whole training pipeline with many epochs in a notebook. While the simple modules (`OhmicPower` and `RadiatedPower`) may work, you will likely experience a crash during the training process if you try to do them all.
>
> If you want to try running this for real, see the script at `modules/transport_predictor/transport_sample_checkpoint.py`

We have found that first training each submodule independently and then using those weights to initialize the full `TransportPredictor` module leads to better results than training the full module from scratch.

Though each submodule has its own data setup scripts and could be used completely independently, when combining modules it is recommended to use the dataloader config from the main module. This ensures that the same data is being used for all modules, and that there isn't data contamination when doing train/val/test splits.

## 2.1 Set up the config

In [ ]:
import jax
import tempfile
from popsim.ml.launch import launch_train
from popsim.ml.trainer import Trainer
from popsim.ml.train_config import TrainConfig

from popsim.modules.transport_predictor.train_configs import BASE_CONFIG, update_submodule_configs

CHECKPOINT_DIR_BASE = tempfile.TemporaryDirectory().name
MAX_EPOCHS = 10
EPOCHS_PER_VAL = 5

transport_predictor_config = BASE_CONFIG.model_copy(
    update={
        "debug": False,
        "max_epochs": MAX_EPOCHS,
        "epochs_per_val": EPOCHS_PER_VAL,
        "checkpoint_dir": os.path.join(CHECKPOINT_DIR_BASE, "transport_predictor"),
        "dataloader_config": {
            **BASE_CONFIG.dataloader_config,
            "ds_path": f"{DATA_DIR}/tcv/scrambled_transport_sample.nc",
            "debug": False,
        },
        "model_init_config": {
            **BASE_CONFIG.model_init_config,
            "restore_submodules": True,
        },
    }
)

# Update all the submodule configs to use the same dataloader config as the main module
transport_predictor_config = update_submodule_configs(
    transport_predictor_config.model_dump(),
    [
        "profile_predictor",
        "power_balance",
        "p_oh_predictor",
        "p_rad_predictor",
    ],
)

## 2.2 Profile Predictor

This should be familiar if you read the profile predictor notebook. Set up the module, loss function, optimizer, and train.

In [ ]:
profile_predictor_config = TrainConfig.load(transport_predictor_config.model_init_config["submodules"]["profile_predictor"])
profile_predictor_config = profile_predictor_config.model_copy(
    update={
        "checkpoint_dir": os.path.join(CHECKPOINT_DIR_BASE, "profile_predictor"),
        "max_epochs": MAX_EPOCHS,
        "epochs_per_val": EPOCHS_PER_VAL,
    }
)
launch_train(profile_predictor_config.model_dump(), use_wandb=False)

## 2.3 Power Balance 

In [ ]:
power_balance_config = TrainConfig.load(transport_predictor_config.model_init_config["submodules"]["power_balance"])
power_balance_config = power_balance_config.model_copy(
    update={
        "max_epochs": MAX_EPOCHS,
        "epochs_per_val": EPOCHS_PER_VAL,
        "checkpoint_dir": os.path.join(CHECKPOINT_DIR_BASE, "power_balance"),
    }
)
launch_train(power_balance_config.model_dump(), use_wandb=False)

## 2.4 OhmicPower and RadiatedPower

These power sources are difficult to predict in advance and measure in real time. Therefore we have modules which put these values in terms of our controllable inputs.

In [ ]:
ohmic_power_config = TrainConfig.load(transport_predictor_config.model_init_config["submodules"]["p_oh_predictor"])
ohmic_power_config = ohmic_power_config.model_copy(
    update={
        "max_epochs": MAX_EPOCHS,
        "epochs_per_val": EPOCHS_PER_VAL,
        "checkpoint_dir": os.path.join(CHECKPOINT_DIR_BASE, "p_oh_predictor"),
    }
)
launch_train(ohmic_power_config.model_dump(), use_wandb=False)

In [ ]:
radiated_power_config = TrainConfig.load(transport_predictor_config.model_init_config["submodules"]["p_rad_predictor"])
radiated_power_config = radiated_power_config.model_copy(
    update={
        "max_epochs": MAX_EPOCHS,
        "epochs_per_val": EPOCHS_PER_VAL,
        "checkpoint_dir": os.path.join(CHECKPOINT_DIR_BASE, "p_rad_predictor"),
    }
)
launch_train(radiated_power_config.model_dump(), use_wandb=False)

## 2.4: Main Module Training

Now that the submodules are trained, we restore the weights as a starting point for the main module training.

The submodule configs contain both their structure (e.g. number of layers, layer width, input size, etc.) as well as the checkpoint directory.
The main module always initializes the submodules according to their structure, and by enabling `restore_submodules` it will fill in the weights from the best saved checkpoint as well.

In [ ]:
transport_predictor_config_final = transport_predictor_config.model_copy(
    update={
        "max_epochs": MAX_EPOCHS,
        "epochs_per_val": EPOCHS_PER_VAL,
        "checkpoint_dir": os.path.join(CHECKPOINT_DIR_BASE, "transport_predictor"),
        "model_init_config": {
            "submodules": {
                "profile_predictor": profile_predictor_config,
                "power_balance": power_balance_config,
                "p_oh_predictor": ohmic_power_config,
                "p_rad_predictor": radiated_power_config,
            },
            "restore_submodules": True,
        }
    }
)
launch_train(transport_predictor_config_final.model_dump(), use_wandb=False)